In [2]:
import pandas as pd
import numpy as np
from google.colab import files
import io

class ISO5259DataQualityAgent:
    """
    Python Agent implementing key data quality measures and frameworks
    aligned with ISO/IEC 5259 (Parts 1-4) for AI and Machine Learning.
    """
    def __init__(self, df: pd.DataFrame, target_column: str = None, protected_attributes: list = None):
        self.df = df
        self.target_column = target_column
        self.protected_attributes = protected_attributes or []
        self.report = {}

    def assess_completeness(self) -> dict:
        """ISO/IEC 5259-2: Completeness - presence of required data attributes."""
        total_cells = self.df.size
        missing_cells = self.df.isnull().sum().sum()
        completeness_score = 1.0 - (missing_cells / total_cells if total_cells > 0 else 0)

        col_completeness = (1.0 - (self.df.isnull().sum() / len(self.df))).to_dict()

        return {
            "dimension": "Completeness",
            "overall_score": float(completeness_score),
            "column_scores": col_completeness
        }

    def assess_uniqueness(self) -> dict:
        """ISO/IEC 5259-2: Uniqueness - detecting duplicate records."""
        total_rows = len(self.df)
        duplicate_rows = self.df.duplicated().sum()
        uniqueness_score = 1.0 - (duplicate_rows / total_rows if total_rows > 0 else 0)

        return {
            "dimension": "Uniqueness",
            "overall_score": float(uniqueness_score),
            "duplicate_count": int(duplicate_rows)
        }

    def assess_validity(self, rules: dict = None) -> dict:
        """ISO/IEC 5259-2: Validity - conformity to types, expected ranges, or logic rules."""
        if not rules:
            return {"dimension": "Validity", "status": "No custom validation rules provided"}

        validity_results = {}
        total_checks = 0
        passed_checks = 0

        for col, condition_func in rules.items():
            if col in self.df.columns:
                passed = self.df[col].apply(condition_func).sum()
                total = len(self.df)
                validity_results[col] = float(passed / total) if total > 0 else 1.0
                total_checks += total
                passed_checks += passed

        overall_validity = float(passed_checks / total_checks) if total_checks > 0 else 1.0
        return {
            "dimension": "Validity",
            "overall_score": overall_validity,
            "attribute_scores": validity_results
        }

    def assess_representativeness_and_bias(self) -> dict:
        """
        ISO/IEC 5259-2 (AI-Specific): Representativeness & Fairness
        Evaluates distribution parity across designated protected/categorical attributes.
        """
        if not self.protected_attributes:
            return {"dimension": "Representativeness", "status": "No protected attributes specified"}

        rep_results = {}
        for attr in self.protected_attributes:
            if attr in self.df.columns:
                value_counts = self.df[attr].value_counts(normalize=True).to_dict()
                proportions = list(value_counts.values())
                min_prop = min(proportions) if proportions else 0
                expected_uniform = 1.0 / len(proportions) if len(proportions) > 0 else 1.0
                balance_score = min_prop / expected_uniform if expected_uniform > 0 else 0

                rep_results[attr] = {
                    "distribution": {str(k): float(v) for k, v in value_counts.items()},
                    "balance_score": float(min(balance_score, 1.0))
                }

        return {
            "dimension": "Representativeness & Fairness",
            "attributes": rep_results
        }

    def run_full_evaluation(self, validity_rules: dict = None) -> dict:
        """Executes the complete ISO/IEC 5259-aligned evaluation suite."""
        print(">[ISO/IEC 5259 Agent] Running Data Quality Assessment...")
        self.report = {
            "dataset_shape": self.df.shape,
            "completeness": self.assess_completeness(),
            "uniqueness": self.assess_uniqueness(),
            "validity": self.assess_validity(validity_rules),
            "representativeness": self.assess_representativeness_and_bias()
        }
        return self.report

    def print_compliance_report(self):
        """Prints a structured summary report meeting ISO/IEC 5259 reporting criteria."""
        if not self.report:
            print("Please execute run_full_evaluation() first.")
            return

        print("\n" + "="*60)
        print(" ISO/IEC 5259 AI DATA QUALITY COMPLIANCE REPORT")
        print("="*60)
        print(f" Dataset Dimensions: {self.report['dataset_shape'][0]} rows | {self.report['dataset_shape'][1]} columns")
        print("-" * 60)
        print(f" [1] Completeness Score : {self.report['completeness']['overall_score']*100:.2f}%")
        print(f" [2] Uniqueness Score   : {self.report['uniqueness']['overall_score']*100:.2f}% ({self.report['uniqueness']['duplicate_count']} duplicates found)")

        val = self.report['validity']
        if "overall_score" in val:
            print(f" [3] Validity Score     : {val['overall_score']*100:.2f}%")
        else:
            print(f" [3] Validity Score     : Skipped ({val.get('status')})")

        print(" [4] Representativeness / Bias Metrics:")
        rep = self.report['representativeness']
        if "attributes" in rep:
            for attr, data in rep['attributes'].items():
                print(f"     - Attribute '{attr}' Balance Score: {data['balance_score']*100:.2f}%")
        else:
            print(f"     - Skipped ({rep.get('status')})")
        print("="*60)

# ==========================================
# GOOGLE COLAB CSV UPLOAD & EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    print("Please upload your CSV file using the prompt below:")
    uploaded = files.upload()

    if uploaded:
        # Grab the first uploaded filename automatically
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
        print(f"\nSuccessfully loaded '{filename}' with shape {df.shape}")
    else:
        print("\nNo file uploaded. Falling back to default sample dataset...")
        data = {
            "age": [25, 32, np.nan, 47, 32, 25, 59],
            "income": [50000, 65000, 120000, 45000, 65000, -100, 85000],
            "region": ["North", "South", "East", "North", "North", "North", "West"],
            "target": [0, 1, 1, 0, 1, 0, 1]
        }
        df = pd.DataFrame(data)

    # Define ISO/IEC 5259-2 validity rules (adjust column names as needed for your CSV)
    validation_rules = {}
    if "age" in df.columns:
        validation_rules["age"] = lambda x: pd.isna(x) or (x > 0 and x < 120)
    if "income" in df.columns:
        validation_rules["income"] = lambda x: pd.isna(x) or (x >= 0)

    # Identify target and protected attributes if present
    target_col = "target" if "target" in df.columns else None
    protected_cols = [col for col in ["region", "gender", "race"] if col in df.columns]

    # Initialize and execute agent
    agent = ISO5259DataQualityAgent(
        df=df,
        target_column=target_col,
        protected_attributes=protected_cols
    )

    agent.run_full_evaluation(validity_rules=validation_rules)
    agent.print_compliance_report()

Please upload your CSV file using the prompt below:



No file uploaded. Falling back to default sample dataset...
>[ISO/IEC 5259 Agent] Running Data Quality Assessment...

 ISO/IEC 5259 AI DATA QUALITY COMPLIANCE REPORT
 Dataset Dimensions: 7 rows | 4 columns
------------------------------------------------------------
 [1] Completeness Score : 96.43%
 [2] Uniqueness Score   : 100.00% (0 duplicates found)
 [3] Validity Score     : 92.86%
 [4] Representativeness / Bias Metrics:
     - Attribute 'region' Balance Score: 57.14%
